# Fundamentals of Data Science - Network Science Lab

# Lab Session 03 - Bipartite Networks

## Bipartite Networks

A bipartite network is a network with two types of nodes and edges that run only between nodes of different kinds.

Assuming the node groups are represented as disjoint sets $V_i$ and $V_j$, a bipartite network can be formally defined as a graph $G = (V, E)$, with $V = V_i \cup V_j$ and $E \subseteq V_i \times V_j$.

In particular, a bipartite network can be summarised using an incidence matrix of size $(V_i \times V_j)$ such that
$$I_{ij} = \begin{cases}
  1 &\quad\text{if item $i$ belongs to group $j$} \\
  0 &\quad\text{otherwise} \\
\end{cases}$$

In [ ]:
import igraph as ig
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path
from itertools import combinations

In [ ]:
base_dir = Path.cwd()
vertex_output_size = 30

In [ ]:
# Define edges for the bipartite network.
bipartite_edges = [
    [0,6], [1,6], [2,6], [2,7], [3,7], [3,8], [4,8], [5,8]
]

# Construct the graph.
g_bip = ig.Graph(edges=bipartite_edges, directed=False)

# Assign the required `type` attribute to designate bipartite sets via Booleans.
g_bip.vs["type"] = [False, False, False, False, False, False, True, True, True]
print("Is bipartite?", g_bip.is_bipartite())

# Plot the network.
fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    g_bip,
    target=ax,
    vertex_label= [x.index for x in g_bip.vs()],
    vertex_label_color = "white",
    vertex_label_size = 10,
    vertex_size= vertex_output_size,
    vertex_color= ["red" if x == False else "blue" for x in g_bip.vs["type"]],
    edge_color= "grey",
    layout = g_bip.layout_bipartite()
)

### The Density of a Bipartite Network

Since edges in a bipartite network can only connect nodes belonging to different groups, the density of a bipartite network will be defined as
$$\delta = \frac{|E|}{|E_\max|} = \frac{|E|}{|V_i| \cdot |V_j|}$$

### One-Mode Projection

Bipartite networks can act as a bridge structure to build unimodal co-occurrence networks that only feature one type of node.

Assuming $I$ is the $(|V_i| \times |V_j|)$ incidence matrix of a bipartite network, it is possible to retrieve co-occurrence by computing one of the following matrix:
- The matrix $A_i = I \times I^T$ represents the projection onto partition $V_i$.
- The matrix $A_j = I^T \times I$ represents the projection onto partition $V_j$.

Overall, performing one-mode projection does not guarantee the creation of simple and unweighted networks, although the procedure always results in the construction of undirected networks.

For this reason, it is possible to obtain the adjacency matrix of a co-occurrence network by performing the following operations:
- Simplifying the network by setting the diagonal elements equal to zero.
- If needed, it is possible to make the network unweighted by setting its non-zero off-diagonal entries equal to one.

In [ ]:
# Extract the incidence matrix and the list of vertices on each partition.
incidence_matrix = g_bip.get_biadjacency()
print("The incidence matrix \n", incidence_matrix_np)

# Perform one-mode projection within each partition.
incidence_matrix_np = np.array(incidence_matrix[0])
projection_left = incidence_matrix_np @ incidence_matrix_np.T
print("Projection on the left side \n", projection_left)

projection_right = incidence_matrix_np.T @ incidence_matrix_np
print("Projection on the right side \n", projection_right)

In [ ]:
# Plot the two possible projections.
fig, ax = plt.subplots(1,2,figsize=(9, 4))

ig.plot(
    g_projection_left,
    target=ax[0],
    vertex_label=list(range(0, g_projection_left.vcount())),
    vertex_size=vertex_output_size,
    vertex_color="red",
    vertex_label_color = "white",
    edge_width=2,
    )
ax[0].set_title("Projection over one mode")

ig.plot(
    g_projection_right,
    target=ax[1],
    vertex_label=list(range(0, g_projection_right.vcount())),
    vertex_size=vertex_output_size,
    vertex_color="blue",
    vertex_label_color = "white",
    edge_width=2,
    )
ax[1].set_title("Projection over the other mode")

Since one-mode projection can be computationally expensive, it is possible to cut down complexity using block-matrix multiplication, which consists of using matrix algebra on smaller sub-matrices, or the clique algorithm, which treats projection as a union of cliques formed by the neighbours of the nodes in the opposite partition.

In [ ]:
# Split the original incidence matrix horizontally into two 3x3 blocks.
I1 = incidence_matrix_np[0:3, :]
I2 = incidence_matrix_np[3:6, :]

# Compute sub-block products independently and assemble them back into the full matrix.
IIT_11 = I1 @ I1.T
IIT_12 = I1 @ I2.T
IIT_21 = I2 @ I1.T
IIT_22 = I2 @ I2.T

IIT_row_blocks = np.block([
    [IIT_11, IIT_12],
    [IIT_21, IIT_22]
])

# Print the resulting projection and the corresponding adjacency matrix.
print("Block projection: \n",IIT_row_blocks)
IIT_row_blocks = binarize_matrix(IIT_row_blocks)
print("Adjacency matrix: \n", IIT_row_blocks)

In [ ]:
# Create a sample bipartite dataset of documents and users.
data = pd.DataFrame({
    'doc_id': ['doc1', 'doc1', 'doc1', 'doc2', 'doc2', 'doc3', 'doc3','doc3'],
    'hashtag': ['Alice', 'Bob', 'Charlie', 'Charlie', 'Dan', 'Dan', 'Eve', 'Frank']
})

# Create a dictionary to store output pairs.
pair_counts = {}

for doc_id, group in data.groupby('doc_id'):
  # Unpack each tuple to retrieve the document ID and the corresponding group-
  print(f"Group Key: {doc_id}")
  print(group[['doc_id', 'hashtag']])

  # Extract unique hashtags for this document and sort them to have cliques of sorted nodes.
  hashtags = sorted(group['hashtag'].unique())
  print(f"Sorted Hashtags: {hashtags}")

  # Generate the pairs.
  pairs = list(combinations(hashtags, 2))
  print(f"Generated Pairs: {pairs}\n")

  # Count pairs, using 0 as a fallback value.
  for pair in pairs:
    pair_counts[pair] = pair_counts.get(pair, 0) + 1


# Convert counts to the final weighted edge list DataFrame.
edge_list = pd.DataFrame(
    [(k[0], k[1], v) for k, v in pair_counts.items()],
    columns=['V1', 'V2', 'weight']
)

print("--- FINAL PROJECTION EDGE LIST ---")
print(edge_list)

In [ ]:
def edge_list_builder(data: pd.DataFrame, col_1: str, col_2: str):
  pair_counts = {}

  # Group by documents and extract unique hashtags per document.
  for _, group in data.groupby(col_1):
    hashtags = sorted(group[col_2].drop_duplicates())

    # Generate all unique pairs of length 2
    for tag1, tag2 in combinations(hashtags, 2):
      pair = (tag1, tag2)
      pair_counts[pair] = pair_counts.get(pair, 0) + 1

  # Convert the dictionary of counts into a structured DataFrame.
  if not pair_counts:
    return pd.DataFrame(columns=['V1', 'V2', 'weight'])

  edgelist_q = pd.DataFrame(
      [(k[0], k[1], v) for k, v in pair_counts.items()],
      columns=['V1', 'V2', 'weight']
  )

  return edgelist_q